# **Importing Libraries**
---
---

In [ ]:
import pandas as pd

import os
from dotenv import load_dotenv

from functions.transformation_functions import merge_weather_pm25_reads, clean_extracted_dataset, normalize_save_pm25_weather_readings
from functions.database_actions_functions import bulk_insert_values_to_db, serve_whole_data

# **Loading Data**
---
---

In [ ]:
path_clean_dataset = os.path.join(os.getcwd(), 'datasets_clean')
if not os.path.exists(path_clean_dataset):
    os.makedirs(path_clean_dataset)
path_datasets_raw = os.path.join(os.getcwd(), 'datasets_raw')

df_sensor_readings = pd.read_csv(os.path.join(path_datasets_raw, 'sensor_readings.csv'))
df_station_sensors = pd.read_csv(os.path.join(path_datasets_raw, 'station_sensors.csv'))
df_weathers_dataset = pd.read_csv(os.path.join(path_datasets_raw, 'weathers_dataset.csv'))

"path_clean_dataset = os.path.join(os.getcwd(), 'datasets_clean')\nif not os.path.exists(path_clean_dataset):\n    os.makedirs(path_clean_dataset)\npath_datasets_raw = os.path.join(os.getcwd(), 'datasets_raw')\n\ndf_sensor_readings = pd.read_csv(os.path.join(path_datasets_raw, 'sensor_readings.csv'))\ndf_station_sensors = pd.read_csv(os.path.join(path_datasets_raw, 'station_sensors.csv'))\ndf_weathers_dataset = pd.read_csv(os.path.join(path_datasets_raw, 'weathers_dataset.csv'))"

In [3]:
df_sensor_readings

,location_id,location_name,sensor_id,values_pm25,timestamp_from,timestamp_to
0,290183.0,Tower Hamlets,2009625.0,6.4,2023-01-02T23:00:00Z,2023-01-03T00:00:00Z
1,290183.0,Tower Hamlets,2009625.0,6.4,2023-01-03T00:00:00Z,2023-01-03T01:00:00Z
2,290183.0,Tower Hamlets,2009625.0,5.4,2023-01-03T01:00:00Z,2023-01-03T02:00:00Z
3,290183.0,Tower Hamlets,2009625.0,5.6,2023-01-03T02:00:00Z,2023-01-03T03:00:00Z
4,290183.0,Tower Hamlets,2009625.0,5.7,2023-01-03T03:00:00Z,2023-01-03T04:00:00Z
...,...,...,...,...,...,...
133934,225751.0,Greenwich,1304607.0,10.8,2026-07-09T18:00:00Z,2026-07-09T19:00:00Z
133935,225751.0,Greenwich,1304607.0,12.5,2026-07-09T19:00:00Z,2026-07-09T20:00:00Z
133936,225751.0,Greenwich,1304607.0,11.8,2026-07-09T20:00:00Z,2026-07-09T21:00:00Z
133937,225751.0,Greenwich,1304607.0,12.2,2026-07-09T21:00:00Z,2026-07-09T22:00:00Z


In [4]:
df_station_sensors

,station_id,station,long,lat,sensor_id
0,270693,Waterloo Place,-0.133068,51.507582,16318059
1,270693,Waterloo Place,-0.133068,51.507582,1561901
2,159,London Westminster,-0.131931,51.494670,28890
3,152,Camden Kerbside,-0.175269,51.544210,235
4,290183,Tower Hamlets,-0.018022,51.502988,2009625
5,290183,Tower Hamlets,-0.018022,51.502988,16318037
6,225751,Greenwich,0.095111,51.486957,1304607
7,225751,Greenwich,0.095111,51.486957,16317995
8,225803,London Honor Oak Park,-0.037418,51.449674,1304734


In [5]:
df_weathers_dataset

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,rain,station_id_london,station_name,long,lat
0,2023-01-01T00:00,12.1,89,34.1,999.1,0.3,270693,Waterloo Place,-0.133068,51.507582
1,2023-01-01T01:00,11.3,94,22.9,999.1,0.1,270693,Waterloo Place,-0.133068,51.507582
2,2023-01-01T02:00,11.9,85,31.1,999.9,0.4,270693,Waterloo Place,-0.133068,51.507582
3,2023-01-01T03:00,11.0,80,31.4,1000.9,0.0,270693,Waterloo Place,-0.133068,51.507582
4,2023-01-01T04:00,10.4,78,30.7,1001.9,0.0,270693,Waterloo Place,-0.133068,51.507582
...,...,...,...,...,...,...,...,...,...,...
185323,2026-07-10T19:00,26.3,36,13.8,1014.3,0.0,225803,London Honor Oak Park,-0.037418,51.449674
185324,2026-07-10T20:00,24.6,46,13.7,1014.4,0.0,225803,London Honor Oak Park,-0.037418,51.449674
185325,2026-07-10T21:00,23.1,56,13.1,1014.7,0.0,225803,London Honor Oak Park,-0.037418,51.449674
185326,2026-07-10T22:00,22.3,61,12.8,1015.2,0.0,225803,London Honor Oak Park,-0.037418,51.449674


# **Checking Anomalies**
---
---

## **1. Combining Weather & PM 2.5 Readings**
---

In [7]:
merged_weather_pm25 = merge_weather_pm25_reads(folder_name='datasets_raw')

In [8]:
merged_weather_pm25

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,rain,station_id_london,station_name,long,lat,location_id,location_name,sensor_id,values_pm25,timestamp_from,timestamp_to
0,2023-01-01 00:00:00,12.1,89,34.1,999.1,0.3,270693,Waterloo Place,-0.133068,51.507582,270693.0,Waterloo Place,1561901.0,5.1,2023-01-01 00:00:00,2023-01-01 01:00:00
1,2023-01-01 01:00:00,11.3,94,22.9,999.1,0.1,270693,Waterloo Place,-0.133068,51.507582,270693.0,Waterloo Place,1561901.0,6.2,2023-01-01 01:00:00,2023-01-01 02:00:00
2,2023-01-01 02:00:00,11.9,85,31.1,999.9,0.4,270693,Waterloo Place,-0.133068,51.507582,270693.0,Waterloo Place,1561901.0,6.6,2023-01-01 02:00:00,2023-01-01 03:00:00
3,2023-01-01 03:00:00,11.0,80,31.4,1000.9,0.0,270693,Waterloo Place,-0.133068,51.507582,270693.0,Waterloo Place,1561901.0,7.9,2023-01-01 03:00:00,2023-01-01 04:00:00
4,2023-01-01 04:00:00,10.4,78,30.7,1001.9,0.0,270693,Waterloo Place,-0.133068,51.507582,270693.0,Waterloo Place,1561901.0,8.3,2023-01-01 04:00:00,2023-01-01 05:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185323,2026-07-10 19:00:00,26.3,36,13.8,1014.3,0.0,225803,London Honor Oak Park,-0.037418,51.449674,NaN,NaN,NaN,NaN,NaT,NaT
185324,2026-07-10 20:00:00,24.6,46,13.7,1014.4,0.0,225803,London Honor Oak Park,-0.037418,51.449674,NaN,NaN,NaN,NaN,NaT,NaT
185325,2026-07-10 21:00:00,23.1,56,13.1,1014.7,0.0,225803,London Honor Oak Park,-0.037418,51.449674,NaN,NaN,NaN,NaN,NaT,NaT
185326,2026-07-10 22:00:00,22.3,61,12.8,1015.2,0.0,225803,London Honor Oak Park,-0.037418,51.449674,NaN,NaN,NaN,NaN,NaT,NaT


## **2. Checking Duplicates & Missing Values**
---

In [9]:
merged_weather_pm25_new = clean_extracted_dataset(folder_name='datasets_raw')
display(merged_weather_pm25_new)

Condition before cleaning:
On merged dataset found 0 duplicated records, missing value details: 
time                        0
temperature_2m              0
relative_humidity_2m        0
wind_speed_10m              0
surface_pressure            0
rain                        0
station_id_london           0
station_name                0
long                        0
lat                         0
location_id             51389
location_name           51389
sensor_id               51389
values_pm25             51516
timestamp_from          51389
timestamp_to            51389
dtype: int64
Stations with > 1 sensors sending values? False
Stations with 0 sensors sending values? False
Stations with inconsistent names? 0

Condition after cleaning:
On merged dataset found 0 duplicated records, missing value details: 
time                        0
temperature_2m              0
relative_humidity_2m        0
wind_speed_10m              0
surface_pressure            0
rain                        0
sta

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,rain,station_id_london,station_name,long,lat,location_id,location_name,sensor_id,values_pm25,timestamp_from,weather_id
0,2023-01-01 00:00:00,12.1,89,34.1,999.1,0.3,270693,Waterloo Place,-0.133068,51.507582,270693,Waterloo Place,1561901,5.1,2023-01-01 00:00:00,W1561901_2023010100
1,2023-01-01 01:00:00,11.3,94,22.9,999.1,0.1,270693,Waterloo Place,-0.133068,51.507582,270693,Waterloo Place,1561901,6.2,2023-01-01 01:00:00,W1561901_2023010101
2,2023-01-01 02:00:00,11.9,85,31.1,999.9,0.4,270693,Waterloo Place,-0.133068,51.507582,270693,Waterloo Place,1561901,6.6,2023-01-01 02:00:00,W1561901_2023010102
3,2023-01-01 03:00:00,11.0,80,31.4,1000.9,0.0,270693,Waterloo Place,-0.133068,51.507582,270693,Waterloo Place,1561901,7.9,2023-01-01 03:00:00,W1561901_2023010103
4,2023-01-01 04:00:00,10.4,78,30.7,1001.9,0.0,270693,Waterloo Place,-0.133068,51.507582,270693,Waterloo Place,1561901,8.3,2023-01-01 04:00:00,W1561901_2023010104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185323,2026-07-10 19:00:00,26.3,36,13.8,1014.3,0.0,225803,London Honor Oak Park,-0.037418,51.449674,225803,London Honor Oak Park,1304734,NaN,2026-07-10 19:00:00,W1304734_2026071019
185324,2026-07-10 20:00:00,24.6,46,13.7,1014.4,0.0,225803,London Honor Oak Park,-0.037418,51.449674,225803,London Honor Oak Park,1304734,NaN,2026-07-10 20:00:00,W1304734_2026071020
185325,2026-07-10 21:00:00,23.1,56,13.1,1014.7,0.0,225803,London Honor Oak Park,-0.037418,51.449674,225803,London Honor Oak Park,1304734,NaN,2026-07-10 21:00:00,W1304734_2026071021
185326,2026-07-10 22:00:00,22.3,61,12.8,1015.2,0.0,225803,London Honor Oak Park,-0.037418,51.449674,225803,London Honor Oak Park,1304734,NaN,2026-07-10 22:00:00,W1304734_2026071022


# **Normalization**
---
---

In [21]:
id_station_detail, merged_weather_pm25_new_v2, station_details = normalize_save_pm25_weather_readings(folder_name_raw='datasets_raw', folder_name_clean='datasets_clean')

Condition before cleaning:
On merged dataset found 0 duplicated records, missing value details: 
time                        0
temperature_2m              0
relative_humidity_2m        0
wind_speed_10m              0
surface_pressure            0
rain                        0
station_id_london           0
station_name                0
long                        0
lat                         0
location_id             51389
location_name           51389
sensor_id               51389
values_pm25             51516
timestamp_from          51389
timestamp_to            51389
dtype: int64
Stations with > 1 sensors sending values? False
Stations with 0 sensors sending values? False
Stations with inconsistent names? 0

Condition after cleaning:
On merged dataset found 0 duplicated records, missing value details: 
time                        0
temperature_2m              0
relative_humidity_2m        0
wind_speed_10m              0
surface_pressure            0
rain                        0
sta

In [22]:
station_details

,station_id,station_name,long,lat
0,270693,Waterloo Place,-0.133068,51.507582
1,159,London Westminster,-0.131931,51.494670
2,152,Camden Kerbside,-0.175269,51.544210
3,290183,Tower Hamlets,-0.018022,51.502988
4,225751,Greenwich,0.095111,51.486957
5,225803,London Honor Oak Park,-0.037418,51.449674


In [23]:
test = id_station_detail.merge(
    station_details,
    left_on='station_id',
    right_on='station_id',
    how='left',
)
test

,sensor_id,station_id,station_name,long,lat
0,16318059,270693,Waterloo Place,-0.133068,51.507582
1,1561901,270693,Waterloo Place,-0.133068,51.507582
2,28890,159,London Westminster,-0.131931,51.494670
3,235,152,Camden Kerbside,-0.175269,51.544210
4,16318037,290183,Tower Hamlets,-0.018022,51.502988
5,2009625,290183,Tower Hamlets,-0.018022,51.502988
6,1304607,225751,Greenwich,0.095111,51.486957
7,16317995,225751,Greenwich,0.095111,51.486957
8,1304734,225803,London Honor Oak Park,-0.037418,51.449674


In [24]:
test_2 = merged_weather_pm25_new_v2.merge(
    test,
    left_on='sensor_id',
    right_on='sensor_id',
    how='left'
)
test_2

,weather_id,sensor_id,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,rain,values_pm25,station_id,station_name,long,lat
0,W1561901_2023010100,1561901,2023-01-01 00:00:00,12.1,89,34.1,999.1,0.3,5.1,270693,Waterloo Place,-0.133068,51.507582
1,W1561901_2023010101,1561901,2023-01-01 01:00:00,11.3,94,22.9,999.1,0.1,6.2,270693,Waterloo Place,-0.133068,51.507582
2,W1561901_2023010102,1561901,2023-01-01 02:00:00,11.9,85,31.1,999.9,0.4,6.6,270693,Waterloo Place,-0.133068,51.507582
3,W1561901_2023010103,1561901,2023-01-01 03:00:00,11.0,80,31.4,1000.9,0.0,7.9,270693,Waterloo Place,-0.133068,51.507582
4,W1561901_2023010104,1561901,2023-01-01 04:00:00,10.4,78,30.7,1001.9,0.0,8.3,270693,Waterloo Place,-0.133068,51.507582
...,...,...,...,...,...,...,...,...,...,...,...,...,...
185323,W1304734_2026071019,1304734,2026-07-10 19:00:00,26.3,36,13.8,1014.3,0.0,NaN,225803,London Honor Oak Park,-0.037418,51.449674
185324,W1304734_2026071020,1304734,2026-07-10 20:00:00,24.6,46,13.7,1014.4,0.0,NaN,225803,London Honor Oak Park,-0.037418,51.449674
185325,W1304734_2026071021,1304734,2026-07-10 21:00:00,23.1,56,13.1,1014.7,0.0,NaN,225803,London Honor Oak Park,-0.037418,51.449674
185326,W1304734_2026071022,1304734,2026-07-10 22:00:00,22.3,61,12.8,1015.2,0.0,NaN,225803,London Honor Oak Park,-0.037418,51.449674


In [25]:
merged_weather_pm25_new_v2

,weather_id,sensor_id,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,rain,values_pm25
0,W1561901_2023010100,1561901,2023-01-01 00:00:00,12.1,89,34.1,999.1,0.3,5.1
1,W1561901_2023010101,1561901,2023-01-01 01:00:00,11.3,94,22.9,999.1,0.1,6.2
2,W1561901_2023010102,1561901,2023-01-01 02:00:00,11.9,85,31.1,999.9,0.4,6.6
3,W1561901_2023010103,1561901,2023-01-01 03:00:00,11.0,80,31.4,1000.9,0.0,7.9
4,W1561901_2023010104,1561901,2023-01-01 04:00:00,10.4,78,30.7,1001.9,0.0,8.3
...,...,...,...,...,...,...,...,...,...
185323,W1304734_2026071019,1304734,2026-07-10 19:00:00,26.3,36,13.8,1014.3,0.0,NaN
185324,W1304734_2026071020,1304734,2026-07-10 20:00:00,24.6,46,13.7,1014.4,0.0,NaN
185325,W1304734_2026071021,1304734,2026-07-10 21:00:00,23.1,56,13.1,1014.7,0.0,NaN
185326,W1304734_2026071022,1304734,2026-07-10 22:00:00,22.3,61,12.8,1015.2,0.0,NaN


In [26]:
merged_weather_pm25_new_v2

,weather_id,sensor_id,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,rain,values_pm25
0,W1561901_2023010100,1561901,2023-01-01 00:00:00,12.1,89,34.1,999.1,0.3,5.1
1,W1561901_2023010101,1561901,2023-01-01 01:00:00,11.3,94,22.9,999.1,0.1,6.2
2,W1561901_2023010102,1561901,2023-01-01 02:00:00,11.9,85,31.1,999.9,0.4,6.6
3,W1561901_2023010103,1561901,2023-01-01 03:00:00,11.0,80,31.4,1000.9,0.0,7.9
4,W1561901_2023010104,1561901,2023-01-01 04:00:00,10.4,78,30.7,1001.9,0.0,8.3
...,...,...,...,...,...,...,...,...,...
185323,W1304734_2026071019,1304734,2026-07-10 19:00:00,26.3,36,13.8,1014.3,0.0,NaN
185324,W1304734_2026071020,1304734,2026-07-10 20:00:00,24.6,46,13.7,1014.4,0.0,NaN
185325,W1304734_2026071021,1304734,2026-07-10 21:00:00,23.1,56,13.1,1014.7,0.0,NaN
185326,W1304734_2026071022,1304734,2026-07-10 22:00:00,22.3,61,12.8,1015.2,0.0,NaN


# **Export to Database**
---

In [27]:
load_dotenv()

list_tables = ['station_details', 'station_sensors', 'weather_readings']
username_postgres = os.getenv('USER_POSTGRES')
pw_postgres = os.getenv('PW_POSTGRES')
db_name = 'data_warehouse_weather'

In [28]:
bulk_insert_values_to_db(username_postgres, pw_postgres, db_name)

Existence of DB data_warehouse_weather: False

Making DB data_warehouse_weather
Existence of DB data_warehouse_weather: True

Tables truncated
Processing for staging.station_details
Completed for staging.station_details
Processing for staging.station_sensors
Completed for staging.station_sensors
Processing for staging.weather_readings
Completed for staging.weather_readings


In [ ]:
_ = serve_whole_data(username_postgres, pw_postgres, db_name, limit=10000)

Existence of DB data_warehouse_weather: True



c:\Users\user\works\DE_studies\openmeteo_ml\database_actions_functions.py:163: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  table_result = pd.io.sql.read_sql_query(f'''


Saved cleaned, read-ML data to c:\Users\user\works\DE_studies\openmeteo_ml\datasets_aggr\weather_readings_mlready.csv
